#📘 Sistemas Baseados em Conhecimento - Aula 4: Motor de Inferência

##🏭 Case: Monitoramento de Impressão 3D Industrial

Hoje a ideia não é “programar um sistema do zero”.

A ideia é **entender como um motor de inferência raciocina**, passo a passo, e como ele consegue **explicar** o motivo de uma decisão.

Ao final, você vai **mudar a política do sistema** adicionando regras e observar como a decisão muda de forma auditável.

> **Ontologia e fatos seguem rigorosamente a Aula 3 (predicados simbólicos).**

## Setup do notebook

**O que vamos fazer:** importar algumas ferramentas básicas (dataclass, tipos).  
**Por que vamos fazer:** isso só prepara o ambiente para o restante do notebook, sem influenciar a lógica do motor.

✅ Execute esta célula uma única vez.

In [35]:
from dataclasses import dataclass
from typing import List, Dict, Set, Optional, Tuple

## Base de Fatos (Aula 3) — a “entrada” do motor

**O que vamos fazer:** carregar os casos F1…F6 como uma lista de fatos simbólicos.  
**Por que vamos fazer:** o motor não começa “do nada”. Ele começa com fatos observados (sensores/grounding).  
Aqui, vamos manter **rigorosamente** a ontologia da Aula 3 (mesmos predicados e valores), porque um motor de inferência depende de consistência.

📌 Pense nisso como o “estado inicial” do mundo para cada caso.

✅ Execute esta célula para ter os fatos disponíveis.

In [36]:
FATOS: Dict[str, List[str]] = {
    "F1": ["evento_visao(stringing)", "confianca(baixa)",  "vibracao(normal)",  "criticidade(media)", "prazo(longo)"],
    "F2": ["evento_visao(layer_shift)", "confianca(alta)", "vibracao(warning)", "criticidade(alta)",  "prazo(curto)"],
    "F3": ["evento_visao(normal)",      "confianca(baixa)", "vibracao(erro)",    "criticidade(media)", "prazo(longo)"],  # conflito
    "F4": ["evento_visao(under_extrusion)", "confianca(baixa)","vibracao(normal)",  "criticidade(baixa)", "prazo(longo)"],
    "F5": ["evento_visao(stringing)",   "confianca(baixa)","vibracao(warning)", "criticidade(alta)",  "prazo(longo)"],
    "F6": ["evento_visao(layer_shift)", "confianca(alta)", "vibracao(normal)",  "criticidade(alta)",  "prazo(longo)"],
}

print("Exemplo F3:", FATOS["F3"])


Exemplo F3: ['evento_visao(normal)', 'confianca(baixa)', 'vibracao(erro)', 'criticidade(media)', 'prazo(longo)']


## O motor formal (mínimo): WorkingMemory + Rule + InferenceEngine

**O que vamos fazer:** definir as 3 peças do motor formal:
- **WorkingMemory (WM):** a “lousa” onde fatos ficam disponíveis
- **Rule:** regra de produção (condições → ação) com prioridade e justificativa
- **InferenceEngine:** executa ciclos **MATCH → SELECT → FIRE**

**Por que vamos fazer:** isso resolve os 3 problemas do motor if/elif:
1) conflitos deixam de ser “ordem do código” e viram **prioridade (política)**
2) surgem **fatos intermediários** (encadeamento)
3) o motor passa a deixar rastros e suportar **explicação**

✅ Execute esta célula uma vez. É a base de tudo.

In [37]:
# Motor Formal (mínimo): WorkingMemory + Rule + InferenceEngine

# dataclass : evita código repetitivo
# frozen : torna os objetos imutável
@dataclass(frozen=True)

# A Rule é um especialista encapsulado.
# Ela sabe quando deve agir (condições),
# o que deve produzir (ação),
# qual sua força na decisão (prioridade),
# e por que existe (justificativa).
class Rule:
    nome: str  #nome único da regra, serve para identificação no log e na explicação (por_que)
    condicoes: List[str] #lista de fatos que precisam estar na WorkingMemory para que a regra possa ser ativada (MATCH) "SE"
    acao: str #fato que será adicionado à WorkingMemory quando a regra for disparada (FIRE) "ENTAO"
    prioridade: int #número usado para resolver conflitos quando duas regras estão ativas ao mesmo tempo, a de maior prioridade vence (SELECT)
    justificativa: str #serve para identificação no log e na explicação (por_que)

# A WorkingMemory é a “lousa” do sistema.
# Ela começa com os fatos observados.
# E vai acumulando novos fatos conforme as regras disparam.
class WorkingMemory:

  #método que recebe os fatos observados inicialmente (vindos do grounding / sensores)
    def __init__(self, fatos_iniciais: List[str]):
        self.fatos: Set[str] = set(fatos_iniciais)

    #método responsável por adicionar um novo fato à memória,retorna True se o fato foi realmente adicionado ou False se ele já existia
    def add(self, fato: str) -> bool:
        if fato in self.fatos:
            return False
        self.fatos.add(fato)
        return True

    # verifica se TODAS as condições de uma regra estão presentes na Working Memory e é usado na etapa MATCH do motor
    def contains_all(self, conds: List[str]) -> bool:
        return all(c in self.fatos for c in conds)

# A InferenceEngine é o “orquestrador” do raciocínio.
# Ela executa o ciclo MATCH → SELECT → FIRE:
# 1) MATCH: encontra quais regras podem disparar (agenda)
# 2) SELECT: escolhe a regra vencedora (ex.: por prioridade)
# 3) FIRE: dispara a regra e adiciona um novo fato na WorkingMemory
# Ela repete até não haver mais regras ativáveis (convergência).
# E registra o caminho (support) para conseguir explicar o "por quê" de uma decisão.
class InferenceEngine:

    # recebe as regras, define a estratégia de conflito (prioridade) e prepara estrutura para rastreabilidade
    def __init__(self, regras: List[Rule]):

        # ordena regras por prioridade (maior primeiro)
        # Isso define a estratégia de resolução de conflito
        self.regras = sorted(regras, key=lambda r: (-r.prioridade, r.nome))

        # Log de regras disparadas (auditoria)
        self.last_fired: List[Tuple[str, str]] = []  # (regra_nome, fato_gerado)

        # estrutura para explicabilidade (por_que)
        self.support: Dict[str, Dict] = {}           # fato -> {"rule":..., "needs":[...]} para por_que()

    # esse método implementa o raciocínio formal:
    def executar(self, wm: WorkingMemory, max_ciclos: int = 30, verbose: bool = True) -> Dict:
        self.last_fired = []
        self.support = {}
        ciclos = 0

        while ciclos < max_ciclos:
            ciclos += 1
            agenda = [r for r in self.regras if wm.contains_all(r.condicoes) and r.acao not in wm.fatos]

            if not agenda:
                if verbose:
                    print(f"\nCiclo {ciclos}: agenda vazia → convergiu.")
                break

            # SELECT: maior prioridade (lista já ordenada)
            escolhida = agenda[0]

            if verbose:
                print(f"\nCiclo {ciclos}")
                print("Agenda:", [a.nome for a in agenda[:6]], "..." if len(agenda) > 6 else "")
                print("SELECT →", escolhida.nome, f"(P={escolhida.prioridade})")
                print("FIRE   →", escolhida.acao)

            # FIRE
            added = wm.add(escolhida.acao)
            if added:
                self.last_fired.append((escolhida.nome, escolhida.acao))
                self.support[escolhida.acao] = {"rule": escolhida, "needs": list(escolhida.condicoes)}

        return {
            "ciclos": ciclos,
            "wm": wm,
            "fired": self.last_fired,
        }

    # reconstrói a cadeia causal, mostra qual regra gerou qual fato, mostra quais fatos sustentaram aquela regra
    def por_que(self, objetivo: str) -> Dict:

        if objetivo not in self.support:
            return {"objetivo": objetivo, "tipo": "observado_ou_ausente", "detalhe": "Fato inicial (observado) ou não derivado pelo motor."}

        node = self.support[objetivo]
        rule: Rule = node["rule"]
        children = []
        for need in node["needs"]:
            children.append(self.por_que(need) if need in self.support else {"objetivo": need, "tipo": "observado", "detalhe": "Fato inicial na WM."})

        return {
            "objetivo": objetivo,
            "tipo": "derivado",
            "regra": rule.nome,
            "prioridade": rule.prioridade,
            "justificativa": rule.justificativa,
            "suportado_por": children,
        }


## Base de Conhecimento (BK) — a política do sistema

**O que vamos fazer:** definir um conjunto mínimo de regras do domínio (BK).  
**Por que vamos fazer:** a BK é a “política” declarada.  
O motor **não é inteligente por si só** — ele é fiel ao que está na BK.

Nas aulas passadas nós criamos a base de conhecimento, base de fatos, regras e encadeamento manual.

Hoje estamos evoluindo esse conhecimento de forma mais estruturada.

📌 Importante:
- Regras geram fatos intermediários (camada 1) e depois decisões (camada 2)
- Conflitos são resolvidos por **prioridade** (critério auditável)
 - O que é Prioridade?
É um critério para resolver conflitos entre regras.
Quando duas regras podem disparar ao mesmo tempo,
a de maior prioridade vence.

✅ Execute para carregar as regras.

In [38]:
# Base de Conhecimento (mínima, alinhada à Aula 3)
# Camada 1: interpretação
# Camada 2: decisão

BASE_REGRAS: List[Rule] = [
    # --- Camada 1 (sensores/visão) ---
    Rule(
        nome="R01_VIBRACAO_ERRO_RISCO_ALTO",
        condicoes=["vibracao(erro)"],
        acao="risco_mecanico(alto)",
        prioridade=95,
        justificativa="Vibração em erro indica anomalia mecânica grave."
    ),
    Rule(
        nome="R02_VIBRACAO_WARNING_RISCO_MEDIO",
        condicoes=["vibracao(warning)"],
        acao="risco_mecanico(medio)",
        prioridade=65,
        justificativa="Warning de vibração indica risco mecânico moderado."
    ),
    Rule(
        nome="R03_LAYER_SHIFT_CONFIRMADO",
        condicoes=["evento_visao(layer_shift)", "confianca(alta)"],
        acao="defeito_confirmado(layer_shift)",
        prioridade=88,
        justificativa="Layer shift com confiança alta é defeito confirmado."
    ),
    Rule(
        nome="R04_STRINGING_SUSPEITO",
        condicoes=["evento_visao(stringing)", "confianca(baixa)"],
        acao="defeito_suspeito(stringing)",
        prioridade=55,
        justificativa="Stringing com confiança baixa é evidência fraca → suspeito."
    ),
    Rule(
    nome="R05_UNDER_EXTRUSION_SUSPEITO",
    condicoes=["evento_visao(under_extrusion)", "confianca(baixa)"],
    acao="defeito_suspeito(under_extrusion)",
    prioridade=55,
    justificativa="Under extrusion com baixa confiança é evidência fraca → tratar como suspeito."
),

    # --- Camada 2 (decisão) ---
    Rule(
        nome="R07_NOGO_LAYER_SHIFT_CRITICO",
        condicoes=["defeito_confirmado(layer_shift)", "criticidade(alta)"],
        acao="decisao(NOGO)",
        prioridade=100,
        justificativa="Defeito confirmado em peça crítica → parar (NOGO)."
    ),
    Rule(
        nome="R08_NOGO_RISCO_MECANICO_ALTO",
        condicoes=["risco_mecanico(alto)"],
        acao="decisao(NOGO)",
        prioridade=97,
        justificativa="Risco mecânico alto → parar por segurança (NOGO)."
    ),
    Rule(
        nome="R09_INVESTIGAR_SUSPEITO_CRITICO",
        condicoes=["defeito_suspeito(stringing)", "criticidade(alta)"],
        acao="decisao(INVESTIGAR)",
        prioridade=68,
        justificativa="Suspeita de defeito em peça crítica → investigar antes de seguir."
    ),
    Rule(
        nome="R10_GO_SENSORES_NORMAIS",
        condicoes=["evento_visao(normal)", "vibracao(normal)"],
        acao="decisao(GO)",
        prioridade=10,
        justificativa="Sem defeitos e vibração normal → seguir (GO)."
    ),
    Rule(
    nome="R11_INVESTIGAR_RISCO_MEDIO_CRITICO",
    condicoes=["risco_mecanico(medio)", "criticidade(alta)"],
    acao="decisao(INVESTIGAR)",
    prioridade=75,
    justificativa="Risco mecânico médio em peça crítica exige investigação antes de seguir."
),

]

print("Regras carregadas:", len(BASE_REGRAS))


Regras carregadas: 10


## Funções auxiliares para rodar casos rapidamente

**O que vamos fazer:** criar funções para:
- rodar um caso (F1…F6) sem repetir código
- exibir um resumo simples (WM final e decisão)

**Por que vamos fazer:** em aula, a gente quer focar no raciocínio do motor, não em “boilerplate”.

✅ Execute esta célula para liberar os atalhos.

In [39]:
# Helpers para rodar casos rapidamente
def rodar_caso(case_id: str, regras: List[Rule] = BASE_REGRAS, verbose: bool = True):
    wm = WorkingMemory(FATOS[case_id])
    eng = InferenceEngine(regras)
    resultado = eng.executar(wm, verbose=verbose)
    return eng, resultado

def mostrar_resumo(case_id: str, resultado: Dict):
    print("\n--- RESUMO ---")
    print("Caso:", case_id)
    print("WM final (ordenada):")
    for f in sorted(list(resultado["wm"].fatos)):
        print(" -", f)
    decis = [f for f in resultado["wm"].fatos if f.startswith("decisao(")]
    print("\nDecisão:", decis[0] if decis else "(nenhuma)")


## 5) Demonstração 1 — Caso F3 (conflito entre sensores)

**O que vamos fazer:** rodar o caso F3, onde:
- a visão diz “normal”
- a vibração diz “erro”

**Por que vamos fazer:** esse é o caso clássico em que um if/elif "ingênuo" falha.  
Aqui vamos observar:
- como o motor constrói uma **agenda** (MATCH)
- como escolhe pela **prioridade** (SELECT)
- como adiciona fatos e converge (FIRE)
- e como consegue responder “por quê?” com `por_que()`

✅ Execute e observe os ciclos e a árvore de justificativa.

In [40]:
# DEMO 1: Caso F3 (conflito: visão normal vs vibração erro)
eng, res = rodar_caso("F3", verbose=True)
mostrar_resumo("F3", res)

# Por quê a decisão?
print("\nÁrvore por_que('decisao(NOGO)'):")
import json
print(json.dumps(eng.por_que("decisao(NOGO)"), indent=2, ensure_ascii=False))



Ciclo 1
Agenda: ['R01_VIBRACAO_ERRO_RISCO_ALTO'] 
SELECT → R01_VIBRACAO_ERRO_RISCO_ALTO (P=95)
FIRE   → risco_mecanico(alto)

Ciclo 2
Agenda: ['R08_NOGO_RISCO_MECANICO_ALTO'] 
SELECT → R08_NOGO_RISCO_MECANICO_ALTO (P=97)
FIRE   → decisao(NOGO)

Ciclo 3: agenda vazia → convergiu.

--- RESUMO ---
Caso: F3
WM final (ordenada):
 - confianca(baixa)
 - criticidade(media)
 - decisao(NOGO)
 - evento_visao(normal)
 - prazo(longo)
 - risco_mecanico(alto)
 - vibracao(erro)

Decisão: decisao(NOGO)

Árvore por_que('decisao(NOGO)'):
{
  "objetivo": "decisao(NOGO)",
  "tipo": "derivado",
  "regra": "R08_NOGO_RISCO_MECANICO_ALTO",
  "prioridade": 97,
  "justificativa": "Risco mecânico alto → parar por segurança (NOGO).",
  "suportado_por": [
    {
      "objetivo": "risco_mecanico(alto)",
      "tipo": "derivado",
      "regra": "R01_VIBRACAO_ERRO_RISCO_ALTO",
      "prioridade": 95,
      "justificativa": "Vibração em erro indica anomalia mecânica grave.",
      "suportado_por": [
        {
        

## Demonstração 2 — Caso F6 (defeito confirmado + peça crítica)

**O que vamos fazer:** rodar o caso F6, onde:
- há layer_shift com confiança alta
- criticidade é alta

**Por que vamos fazer:** esse caso mostra a ideia de **regra mais forte / política mais conservadora**.  
Ele também ilustra como a decisão é:
- consistente
- auditável
- explicável (árvore de suporte)

✅ Execute e compare o “por quê” do F6 com o do F3.

In [41]:
# DEMO 2: Caso F6 (layer_shift confirmado + criticidade alta → NOGO pela regra mais específica)
eng, res = rodar_caso("F6", verbose=False)
mostrar_resumo("F6", res)

import json
print("\nÁrvore por_que('decisao(NOGO)') no F6:")
print(json.dumps(eng.por_que("decisao(NOGO)"), indent=2, ensure_ascii=False))



--- RESUMO ---
Caso: F6
WM final (ordenada):
 - confianca(alta)
 - criticidade(alta)
 - decisao(NOGO)
 - defeito_confirmado(layer_shift)
 - evento_visao(layer_shift)
 - prazo(longo)
 - vibracao(normal)

Decisão: decisao(NOGO)

Árvore por_que('decisao(NOGO)') no F6:
{
  "objetivo": "decisao(NOGO)",
  "tipo": "derivado",
  "regra": "R07_NOGO_LAYER_SHIFT_CRITICO",
  "prioridade": 100,
  "justificativa": "Defeito confirmado em peça crítica → parar (NOGO).",
  "suportado_por": [
    {
      "objetivo": "defeito_confirmado(layer_shift)",
      "tipo": "derivado",
      "regra": "R03_LAYER_SHIFT_CONFIRMADO",
      "prioridade": 88,
      "justificativa": "Layer shift com confiança alta é defeito confirmado.",
      "suportado_por": [
        {
          "objetivo": "evento_visao(layer_shift)",
          "tipo": "observado",
          "detalhe": "Fato inicial na WM."
        },
        {
          "objetivo": "confianca(alta)",
          "tipo": "observado",
          "detalhe": "Fato inicial

## Agora é com vocês — prática

Até aqui, nós só observamos o motor raciocinar.  
Agora vocês vão fazer o que realmente importa em sistemas baseados em conhecimento:

✅ **alterar a política** adicionando regras.

📌 Reforço:
- NÃO mexer no motor
- O motor só executa
- Quem “define a decisão” é a Base de Conhecimento (regras)

➡️ Você só vai **adicionar regras** (o motor já está pronto).

In [42]:
# EXERCÍCIO: implemente as 2 regras abaixo (TODO) e teste no F5

REGRAS_EXT = BASE_REGRAS.copy()

# TODO 1: criar regra de alerta preventivo (risco_mecanico(medio) + criticidade(alta) -> alerta_preventivo(ativo))
regra_alerta = Rule(
    nome="R12_ALERTA_PREVENTIVO_RISCO_MEDIO_CRITICO",
    condicoes=["defeito_suspeito(stringing)", "criticidade(alta)"],
    acao="alerta_preventivo(ativo)",
    prioridade=85,
    justificativa="Risco médio em peça crítica → ativar alerta preventivo."
)

# TODO 2: criar regra de decisão (alerta_preventivo(ativo) + defeito_suspeito(stringing) -> decisao(INVESTIGAR))
regra_investigar = Rule(
    nome="R13_INVESTIGAR_ALERTA_PREVENTIVO",
    condicoes=["alerta_preventivo(ativo)", "defeito_suspeito(stringing)"],
    acao="decisao(INVESTIGAR)",
    prioridade=90,
    justificativa="Alerta preventivo + suspeita de defeito → investigar."
)

REGRAS_EXT.extend([regra_alerta, regra_investigar])

# Rodar F5 antes/depois (aqui já está 'depois' com REGRAS_EXT)
eng_ext, res_ext = rodar_caso("F5", regras=REGRAS_EXT, verbose=True)
mostrar_resumo("F5", res_ext)

import json
print("\nÁrvore por_que('decisao(INVESTIGAR)'):")
print(json.dumps(eng_ext.por_que("decisao(INVESTIGAR)"), indent=2, ensure_ascii=False))



Ciclo 1
Agenda: ['R02_VIBRACAO_WARNING_RISCO_MEDIO', 'R04_STRINGING_SUSPEITO'] 
SELECT → R02_VIBRACAO_WARNING_RISCO_MEDIO (P=65)
FIRE   → risco_mecanico(medio)

Ciclo 2
Agenda: ['R11_INVESTIGAR_RISCO_MEDIO_CRITICO', 'R04_STRINGING_SUSPEITO'] 
SELECT → R11_INVESTIGAR_RISCO_MEDIO_CRITICO (P=75)
FIRE   → decisao(INVESTIGAR)

Ciclo 3
Agenda: ['R04_STRINGING_SUSPEITO'] 
SELECT → R04_STRINGING_SUSPEITO (P=55)
FIRE   → defeito_suspeito(stringing)

Ciclo 4
Agenda: ['R12_ALERTA_PREVENTIVO_RISCO_MEDIO_CRITICO'] 
SELECT → R12_ALERTA_PREVENTIVO_RISCO_MEDIO_CRITICO (P=85)
FIRE   → alerta_preventivo(ativo)

Ciclo 5: agenda vazia → convergiu.

--- RESUMO ---
Caso: F5
WM final (ordenada):
 - alerta_preventivo(ativo)
 - confianca(baixa)
 - criticidade(alta)
 - decisao(INVESTIGAR)
 - defeito_suspeito(stringing)
 - evento_visao(stringing)
 - prazo(longo)
 - risco_mecanico(medio)
 - vibracao(warning)

Decisão: decisao(INVESTIGAR)

Árvore por_que('decisao(INVESTIGAR)'):
{
  "objetivo": "decisao(INVESTIGAR

### Desafio rápido
Aumente/diminua a prioridade de `R12_INVESTIGAR_ALERTA_PREVENTIVO` e observe se muda a decisão/agenda.
**prioridade é critério de negócio** (auditável).

## Parte 2 — EXERCÍCIO (editar aqui)

Nova política após incidente real:

1) `risco_mecanico(medio)` + `criticidade(alta)` → `alerta_preventivo(ativo)`  
2) `alerta_preventivo(ativo)` + `defeito_suspeito(stringing)` → `decisao(INVESTIGAR)`

Complete as regras abaixo preenchendo:

- `condicoes`
- `acao`
- `prioridade`

⚠ **NÃO alterar o motor.**

---

### 💡 DICA

- Leia a **justificativa**.
- Identifique quais **fatos precisam existir**.
- Compare com as **prioridades já existentes** antes de escolher um número.

Transforme a justificativa em:

- `condicoes = [...]`
- `acao = "..."`
- `prioridade = ?`

> A prioridade deve ser escolhida pensando em **qual regra deve vencer em caso de conflito**.

In [43]:

# ===== IMPLEMENTAR AQUI =====

REGRAS_EXT = BASE_REGRAS.copy()

regra_alerta = Rule(
    nome="R12_ALERTA_PREVENTIVO_RISCO_MEDIO_CRITICO",
    condicoes=[
        "risco_mecanico(medio)", "criticidade(alta)"
    ],
    acao="alerta_preventivo(ativo)",
    prioridade=85,
    justificativa="Risco médio em peça crítica → ativar alerta preventivo."
)

regra_investigar = Rule(
    nome="R13_INVESTIGAR_ALERTA_PREVENTIVO",
    condicoes=[
        "alerta_preventivo(ativo)", "defeito_suspeito(stringing)"
    ],
    acao="decisao(INVESTIGAR)",
    prioridade=90,
    justificativa="Alerta preventivo + suspeita de defeito → investigar."
)

REGRAS_EXT.extend([regra_alerta, regra_investigar])

eng_ext, res_ext = rodar_caso("F5", regras=REGRAS_EXT, verbose=True)
mostrar_resumo("F5", res_ext)

import json
print(json.dumps(eng_ext.por_que("decisao(INVESTIGAR)"), indent=2, ensure_ascii=False))



Ciclo 1
Agenda: ['R02_VIBRACAO_WARNING_RISCO_MEDIO', 'R04_STRINGING_SUSPEITO'] 
SELECT → R02_VIBRACAO_WARNING_RISCO_MEDIO (P=65)
FIRE   → risco_mecanico(medio)

Ciclo 2
Agenda: ['R12_ALERTA_PREVENTIVO_RISCO_MEDIO_CRITICO', 'R11_INVESTIGAR_RISCO_MEDIO_CRITICO', 'R04_STRINGING_SUSPEITO'] 
SELECT → R12_ALERTA_PREVENTIVO_RISCO_MEDIO_CRITICO (P=85)
FIRE   → alerta_preventivo(ativo)

Ciclo 3
Agenda: ['R11_INVESTIGAR_RISCO_MEDIO_CRITICO', 'R04_STRINGING_SUSPEITO'] 
SELECT → R11_INVESTIGAR_RISCO_MEDIO_CRITICO (P=75)
FIRE   → decisao(INVESTIGAR)

Ciclo 4
Agenda: ['R04_STRINGING_SUSPEITO'] 
SELECT → R04_STRINGING_SUSPEITO (P=55)
FIRE   → defeito_suspeito(stringing)

Ciclo 5: agenda vazia → convergiu.

--- RESUMO ---
Caso: F5
WM final (ordenada):
 - alerta_preventivo(ativo)
 - confianca(baixa)
 - criticidade(alta)
 - decisao(INVESTIGAR)
 - defeito_suspeito(stringing)
 - evento_visao(stringing)
 - prazo(longo)
 - risco_mecanico(medio)
 - vibracao(warning)

Decisão: decisao(INVESTIGAR)
{
  "objeti

## Fechamento

Hoje vimos duas ideias centrais:

1) **Forward chaining:** fatos → decisão  
2) **Explicabilidade (`por_que`)**: decisão → fatos + regras

Na próxima aula, vamos atacar um limite do mundo binário:
- Qual é o limiar de “confiança alta”?
- 0.85 e 0.84 mudam tudo?


## Pergunta final (discursiva)

Explique em 5–8 linhas:

1) O motor é “inteligente” ou apenas consistente?  
2) O que garante que a decisão seja auditável?  
3) Se invertermos prioridades, o que muda conceitualmente?

> Dica: use as palavras “política”, “prioridade”, “rastreabilidade” e “justificativa”.